In [12]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [13]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [35]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    # CHANGED: system prompt enforces raw JSON — replaces prefill + stop sequence
    system = "You are a dataset generator. Always respond with raw, valid JSON only — no markdown fences, no explanation, no preamble."

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages, system=system)
    
     
    # Strip markdown fences if model adds them despite system prompt
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]        # remove opening ```json line
        text = text.rsplit("```", 1)[0]      # remove closing ```
        text = text.strip()
    
    return json.loads(text)

In [36]:
dataset = generate_dataset()

dataset

[{'task': 'Extract all IAM role ARNs from an AWS CloudFormation template output',
  'format': 'regex',
  'solution_criteria': 'The regex should match ARNs with the pattern arn:aws:iam::[0-9]{12}:role/[a-zA-Z0-9_-]+ and capture the role name portion'},
 {'task': 'Parse an AWS S3 bucket policy JSON and return a list of all allowed AWS principals',
  'format': 'python',
  'solution_criteria': 'The function should extract the Principal field from each statement in the policy document and return unique principals as a list'},
 {'task': 'Generate a valid AWS Lambda environment variables JSON object with keys for database connection parameters',
  'format': 'json',
  'solution_criteria': 'The JSON should include DB_HOST, DB_PORT, DB_USER, and DB_PASSWORD as keys with reasonable example values, with at least one value containing a complex string'}]

In [37]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [38]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    # CHANGED: system prompt enforces raw JSON — replaces prefill + stop sequence
    system = "You are a code evaluation assistant. Always respond with raw, valid JSON only — no markdown fences, no explanation, no preamble."

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages, system=system)

    # Strip markdown fences if model adds them despite system prompt
    eval_text = eval_text.strip()
    if eval_text.startswith("```"):
        eval_text = eval_text.split("\n", 1)[1]        # remove opening ```json line
        eval_text = eval_text.rsplit("```", 1)[0]      # remove closing ```
        eval_text = eval_text.strip()

    return json.loads(eval_text)

In [39]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    # CHANGED: system prompt enforces raw code output — replaces prefill + stop sequence
    system = "You are a code generation assistant. Respond only with raw Python, JSON, or a plain Regex — no markdown fences, no comments, no explanation."

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages, system=system)
    return output

In [40]:
# Functions to validate the output structure (unchanged)
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    fmt = test_case["format"]
    if fmt == "json":
        return validate_json(response)
    elif fmt == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [41]:
# Function to execute a single test case and grade the output (unchanged)
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [42]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [43]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 2.5


In [44]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n```python\nimport json\nimport re\nimport sys\n\ndef extract_iam_role_arns(cloudformation_template):\n    \"\"\"\n    Extract all IAM role ARNs from an AWS CloudFormation template.\n    \n    Args:\n        cloudformation_template: Either a string (JSON/YAML) or dict representing the template\n        \n    Returns:\n        List of IAM role ARNs found in the template\n    \"\"\"\n    # Parse the template if it's a string\n    if isinstance(cloudformation_template, str):\n        try:\n            template = json.loads(cloudformation_template)\n        except json.JSONDecodeError:\n            # Try to handle YAML-like format by treating as raw string\n            template = cloudformation_template\n    else:\n        template = cloudformation_template\n    \n    iam_role_arns = []\n    \n    if isinstance(template, dict):\n        # Extract from Resources section\n        if 'Resources' in template:\n            resources = template['Resources']\n            for